In [ ]:
from pyspark.sql import functions as sf
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime


In [ ]:

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_data_warehouse"
CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog("reporting")

spark

In [ ]:
from pyspark.sql import functions as sf


df = spark.range(1)

# Generate a sequence of numbers from 0 to (30 * 365)+7 = 10957
df = df.withColumn(
    "numbers", 
    sf.sequence(start=sf.lit(1), stop=sf.lit((30 * 365)+7), step=sf.lit(1))
)

# Explode the array into rows
df_exploded = df.select(sf.explode("numbers").alias("datekey"))

# --- Using sf.date_add() for Date Arithmetic ---
df_exploded = df_exploded.withColumn(
    "datevalue",
    sf.date_add(sf.lit("2010-01-01"), sf.col("datekey")-1)
)


# --- Using sf. for other Date Parts ---
df_exploded = df_exploded\
    .withColumn("datecode", sf.date_format(sf.col("datevalue"), "yyyyMMdd"))\
    .withColumn("yearmonth_code", sf.date_format(sf.col("datevalue"), "yyyyMM"))\
    .withColumn("day_number", sf.day("datevalue"))\
    .withColumn("day_name", sf.dayname("datevalue"))\
    .withColumn("day_name_full", sf.date_format(sf.col("datevalue"), "EEEE"))\
    .withColumn("short_month_name", sf.monthname("datevalue"))\
    .withColumn("full_month_name", sf.date_format(sf.col("datevalue"), "MMMM"))\
    .withColumn("calendar_month_number", sf.date_format(sf.col("datevalue"), "M").cast("int"))\
    .withColumn("calendar_month_label", sf.date_format(sf.col("datevalue"), "yyyyMMM"))\
    .withColumn("calendar_year", sf.date_format(sf.col("datevalue"), "yyyy"))

# Display the first few rows
df_exploded.printSchema()

df_exploded.filter(
    (sf.col("datevalue") >= sf.lit("2025-09-25")) &
    (sf.col("datevalue") < sf.lit("2025-10-05"))   
).show(20, truncate=False)


In [ ]:
# sql_dimension_dates = "DROP TABLE IF EXISTS reporting.dimension.dates"

sql_dimension_dates = "CREATE TABLE IF NOT EXISTS reporting.dimension.dates ( "\
    "datekey integer, "\
    "datevalue date, "\
    "datecode string, "\
    "yearmonth_code string, "\
    "day_number integer, "\
    "day_name string, "\
    "day_name_full string, "\
    "short_month_name string, "\
    "full_month_name string, "\
    "calendar_month_number integer, "\
    "calendar_month_label string, "\
    "calendar_year string, "\
    "year INT, "\
    "month INT "\
        ") "\
"USING ICEBERG "\
"PARTITIONED BY (calendar_year) "\
"TBLPROPERTIES ('comment' = 'dimension table for 30 Years Date')"



spark.sql(sql_dimension_dates)



In [ ]:
# df_exploded = df_exploded \
#     .withColumn("year", sf.year("datevalue")) \
#     .withColumn("month", sf.month("datevalue"))

df_exploded.writeTo("reporting.dimension.dates") \
    .partitionedBy("calendar_year") \
    .using("iceberg") \
    .createOrReplace()

In [ ]:
spark.table("reporting.dimension.dates").show(10, truncate=False)

In [ ]:
table_name = "reporting.dimension.dates"
spark.sql("SHOW TBLPROPERTIES " + table_name).show(truncate=False)
# spark.sql("DESCRIBE TABLE EXTENDED " + table_name).show(truncate=False)
# spark.read.table(table_name + ".partitions").show(truncate=False)

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED " + table_name).show(truncate=False)


In [ ]:
spark.read.table(table_name + ".partitions").show(40, truncate=False)

In [29]:
spark.stop()